In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

c:\Users\Shane\Desktop\Other\vscode\Github Projects\iCWXnlHINuPDOzFR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read in the data
df = pd.read_csv("../data/data.csv")

#format job titles
df['job_title'] = df['job_title'].str.replace('[^a-zA-Z0-9\\s]', '', regex=True).str.lower()
df['job_title'] = df['job_title'].str.replace(r'[\r\n\s]+', ' ', regex=True)

In [3]:
# -------------------------
# Load embedding model
# -------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5444.40it/s]


In [4]:
#create new dataframe with needed columns
structured_df = df[["id","job_title"]].copy()

structured_df

,id,job_title
0,1,2019 ct bauer college of business graduate mag...
1,2,native english teacher at epik english program...
2,3,aspiring human resources professional
3,4,people development coordinator at ryan
4,5,advisory board member at celal bayar university
...,...,...
99,100,aspiring human resources manager graduating ma...
100,101,human resources generalist at loparex
101,102,business intelligence and analytics at travelers
102,103,always set them up for success


In [5]:
#adding job field column to structured_df

#list of jobs
job_field_mapping = {
    # Technology
    "Software Developer": "technology",
    "Software Engineer": "technology",
    "Frontend Developer": "technology",
    "Backend Developer": "technology",
    "Full Stack Developer": "technology",
    "Systems Administrator": "technology",
    "Business Intelligence Analyst": "technology",
    "Data Engineer": "technology",

    # Finance
    "Financial Analyst": "finance",
    "Accountant": "finance",
    "Financial Advisor": "finance",

    # Human Resources
    "Human Resources Manager": "human resources",
    "HR Specialist": "human resources",
    "HR Coordinator": "human resources",
    "Recruiter": "human resources",
    "Talent Acquisition Specialist": "human resources",
    "Human Resources Generalist": "human resources",

    # Customer Service
    "Customer Service Representative": "customer service",

    # Engineering
    "Civil Engineer": "engineering",
    "Technician": "engineering",
    "Field Service Technician": "engineering",
    "Construction Manager": "engineering",

    # Education
    "Teacher": "education",
    "English Teacher": "education",
    "Professor": "education",
    "Lecturer": "education",
    "Teaching Assistant": "education",
    "Admissions Representative": "education",

    # Sales and Marketing
    "Communications Specialist": "sales and marketing",
    "Marketing Officer": "sales and marketing",
    "Real Estate Agent": "sales and marketing",
    "Entrepreneur": "sales and marketing",

    # Operations
    "Business Analyst": "operations",
    "Project Manager": "operations",
    "Operations Analyst": "operations",
    "Logistics Manager": "operations",

    # Healthcare
    "Physician": "healthcare",
    "Healthcare Analyst": "healthcare",

    # Other
    "Research Assistant": "other",
    "Research Scientist": "other",
    "Student": "other",
    "Unemployed": "other"
}





# create new feature columns for identified job title and similarity score
concise_titles = list(job_field_mapping.keys())
concise_title_embeddings = model.encode(concise_titles, normalize_embeddings=True)
job_title_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)

concise_title_similarities = job_title_embeddings @ concise_title_embeddings.T
best_title_index = concise_title_similarities.argmax(axis=1)
best_title_score = concise_title_similarities.max(axis=1)

structured_df["concise_job_title"] = [concise_titles[i] if score >= 0.40 else "Other" for i, score in zip(best_title_index, best_title_score)]
structured_df["job_sector"] = structured_df["concise_job_title"].map(job_field_mapping).fillna("other")

#show all columns in the dataframe
pd.set_option("display.max_rows", None)
structured_df

,id,job_title,concise_job_title,job_sector
0,1,2019 ct bauer college of business graduate mag...,Human Resources Manager,human resources
1,2,native english teacher at epik english program...,English Teacher,education
2,3,aspiring human resources professional,Human Resources Manager,human resources
3,4,people development coordinator at ryan,HR Coordinator,human resources
4,5,advisory board member at celal bayar university,Admissions Representative,education
5,6,aspiring human resources specialist,Human Resources Manager,human resources
6,7,student at humber college and aspiring human r...,Human Resources Generalist,human resources
7,8,hr senior specialist,HR Specialist,human resources
8,9,student at humber college and aspiring human r...,Human Resources Generalist,human resources
9,10,seeking human resources hris and generalist po...,Human Resources Generalist,human resources


In [6]:
# address location column to structured_df

df["location"] = df["location"].str.replace(r"Kanada$", "Canada", regex=True).str.strip()
has_comma = df["location"].str.contains(",", na=False)

split_location = (df.loc[has_comma, "location"].str.split(",", n=1, expand=True))

df["city"] = pd.NA
df["region"] = pd.NA

df.loc[has_comma, "city"] = split_location[0].str.strip()
df.loc[has_comma, "region"] = split_location[1].str.strip()

is_region = ~df["location"].str.contains(",", na=False)
df.loc[is_region, "region"] = df.loc[is_region, "location"]


structured_df["city"] = df["city"]
structured_df["region"] = df["region"]


In [7]:
# add seniority column to structured_df
#identifying seniority levels in job titles
seniority_levels = ["Junior", "Mid", "Senior"]

senior_titles = [
    "senior professional",
    "experienced professional",
    "lead professional",
    "team lead",
    "manager",
    "senior manager",
    "director",
    "principal",
    "head of department",
    "executive"
]

junior_titles = [
    "junior",
    "junior professional",
    "intern",
    "student",
    "entry level",
    "entry level professional",
    "graduate",
    "recent graduate",
    "undergraduate",
    "undergraduate student",
    "undergraduate researcher",
    "trainee",
    "aspiring",
    "assisstant",
    "aspiring professional",    
]

senior_embeddings = model.encode(senior_titles, normalize_embeddings=True)
junior_embeddings = model.encode(junior_titles, normalize_embeddings=True)

job_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)

senior_scores = job_embeddings @ senior_embeddings.T
junior_scores = job_embeddings @ junior_embeddings.T

senior_similarity = senior_scores.max(axis=1)
junior_similarity = junior_scores.max(axis=1)

#add connections as considerations for seniority
df["connections"] = (df["connection"].str.replace("+", "", regex=False).astype("Int64"))
senior_connection_bonus = np.select([df["connections"] >= 500, df["connections"] >= 200], [0.08, 0.02], default=0)
junior_connection_bonus = np.select([df["connections"] <= 100, df["connections"] < 500], [0.08, 0.02], default=0)


adjusted_senior_similarity = senior_similarity + senior_connection_bonus
adjusted_junior_similarity = junior_similarity + junior_connection_bonus

df["adjusted_senior_similarity"] = adjusted_senior_similarity
df["adjusted_junior_similarity"] = adjusted_junior_similarity
threshold = 0.50

structured_df["seniority"] = np.select(
    [
        (adjusted_senior_similarity >= threshold) & (adjusted_senior_similarity > adjusted_junior_similarity),
        (adjusted_junior_similarity >= threshold) & (adjusted_junior_similarity > adjusted_senior_similarity)
    ],
    ["senior", "junior"],
    default="mid"
)

In [9]:
# adding target column "fit" to structured_df
structured_df["fit"] = df["fit"]

#show all columns in the structured_df
pd.reset_option("display.max_rows")
#pd.set_option("display.max_rows", None)
structured_df

,id,job_title,concise_job_title,job_sector,city,region,seniority,fit
0,1,2019 ct bauer college of business graduate mag...,Human Resources Manager,human resources,Houston,Texas,junior,NaN
1,2,native english teacher at epik english program...,English Teacher,education,<NA>,Canada,mid,NaN
2,3,aspiring human resources professional,Human Resources Manager,human resources,Raleigh-Durham,North Carolina Area,junior,NaN
3,4,people development coordinator at ryan,HR Coordinator,human resources,Denton,Texas,mid,NaN
4,5,advisory board member at celal bayar university,Admissions Representative,education,İzmir,Türkiye,mid,NaN
...,...,...,...,...,...,...,...,...
99,100,aspiring human resources manager graduating ma...,Human Resources Manager,human resources,Cape Girardeau,Missouri,mid,NaN
100,101,human resources generalist at loparex,Human Resources Generalist,human resources,Raleigh-Durham,North Carolina Area,mid,NaN
101,102,business intelligence and analytics at travelers,Business Intelligence Analyst,technology,<NA>,Greater New York City Area,mid,NaN
102,103,always set them up for success,Other,other,<NA>,Greater Los Angeles Area,mid,NaN


In [ ]:
#model training phase

# =========================
# 1. Precompute embeddings
# =========================

concise_embeddings = model.encode(structured_df["concise_job_title"].fillna("Other").tolist(), normalize_embeddings=True, show_progress_bar=True)
original_embeddings = model.encode(structured_df["job_title"].fillna("").tolist(), normalize_embeddings=True, show_progress_bar=True)

# Precompute embeddings for standardized titles
concise_titles = list(job_field_mapping.keys())
query_title_embeddings = model.encode(concise_titles, normalize_embeddings=True)

# Store sector for each standardized title in the same order
query_title_sectors = [job_field_mapping[title] for title in concise_titles]


# =========================
# 2. Semantic search
# =========================

def semantic_search(query, df, concise_embeddings, original_embeddings, query_title_embeddings, query_title_sectors, model, threshold=0.40, top_k=10):

    # Embed recruiter query once
    query_embedding = model.encode(query, normalize_embeddings=True)

    # Calculate similarities
    concise_similarity = concise_embeddings @ query_embedding
    original_similarity = original_embeddings @ query_embedding

    # Infer query sector using the closest standardized title
    query_title_scores = query_title_embeddings @ query_embedding
    query_title_index = query_title_scores.argmax()
    query_sector = query_title_sectors[query_title_index]

    # Compare candidate sectors with query sector
    sector_match = (df["job_sector"].fillna("other").str.lower().to_numpy() == query_sector.lower()).astype(float)

    # Combine ranking scores
    final_score = 0.7 * concise_similarity + 0.2 * original_similarity + 0.1 * sector_match

    # Retrieve candidates above the threshold, then select top K
    eligible_candidates = np.where(final_score >= threshold)[0]
    top_indices = eligible_candidates[np.argsort(final_score[eligible_candidates])[::-1][:top_k]]
    results = df.iloc[top_indices].copy()

    # Add score breakdown
    results["concise_similarity"] = concise_similarity[top_indices]
    results["original_similarity"] = original_similarity[top_indices]
    results["sector_match"] = sector_match[top_indices]
    results["fit"] = final_score[top_indices]

    return results


# =========================
# 3. Test
# =========================

results = semantic_search(query="software engineer", 
                          df=structured_df, 
                          concise_embeddings=concise_embeddings, 
                          original_embeddings=original_embeddings, 
                          query_title_embeddings=query_title_embeddings, 
                          query_title_sectors=query_title_sectors, 
                          model=model, 
                          threshold=0.50, 
                          top_k=100
                          )

pd.set_option("display.max_rows", None)
results

Batches: 100%|██████████| 4/4 [00:00<00:00, 35.53it/s]


,id,job_title,concise_job_title,job_sector,city,region,seniority,fit,concise_similarity,original_similarity,sector_match
85,86,information systems specialist and programmer ...,Data Engineer,technology,Gaithersburg,Maryland,junior,0.731827,0.740648,0.566867,1.0
79,80,junior mes engineer information systems,Data Engineer,technology,Myrtle Beach,South Carolina Area,junior,0.719935,0.740648,0.507405,1.0


In [17]:
def candidate_feedback_similarity(df, candidate_embeddings, starred_positions):
    embeddings = np.asarray(candidate_embeddings, dtype=float)
    starred_positions = list(dict.fromkeys(starred_positions))

    weights = {
        "title": 0.45,
        "sector": 0.20,
        "seniority": 0.15,
        "city": 0.10,
        "region": 0.10
    }

    # 1. Job-title embedding similarity
    title_sim = embeddings @ embeddings[starred_positions].T

    # 2. Seniority similarity
    seniority_order = {"junior": 1, "mid": 2, "senior": 3}

    seniority = df["seniority"].astype("string").str.lower().map(seniority_order).to_numpy(dtype=float, na_value=np.nan)
    star_seniority = seniority[starred_positions]
    seniority_sim = 1 - np.abs(seniority[:, None] - star_seniority[None, :]) / 2
    seniority_valid = np.isfinite(seniority[:, None]) & np.isfinite(star_seniority[None, :])
    seniority_sim = np.where(seniority_valid, seniority_sim, 0)

    # 3. Categorical matches, ignoring missing values
    features = {}
    valid = {}

    for col, key in [("job_sector", "sector"), ("city", "city"), ("region", "region")]:
        values = df[col].astype("string").str.strip().str.lower()
        values = values.mask(values.eq(""))
        values = values.astype(object).where(values.notna(), None).to_numpy()
        star_values = values[starred_positions]

        valid[key] = pd.notna(values[:, None]) & pd.notna(star_values[None, :])
        features[key] = ((values[:, None] == star_values[None, :]) & valid[key]).astype(float)

    # 4. Combine feature similarities, excluding unavailable features
    similarities = {
        "title": title_sim,
        "sector": features["sector"],
        "seniority": seniority_sim,
        "city": features["city"],
        "region": features["region"]
    }

    valid_features = {
        "title": np.ones_like(title_sim, dtype=bool),
        "sector": valid["sector"],
        "seniority": seniority_valid,
        "city": valid["city"],
        "region": valid["region"]
    }

    numerator = sum(
        weights[key] * similarities[key] * valid_features[key]
        for key in weights
    )
    denominator = sum(
        weights[key] * valid_features[key]
        for key in weights
    )

    pairwise_scores = np.divide(
        numerator, denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0
    )

    # 5. Compare each candidate to their closest starred example
    feedback_scores = pairwise_scores.max(axis=1)

    #re-ranked score 
    re_ranked_score = 0.7 * df["fit"] + 0.3 * feedback_scores

    # Keep starred candidates at the top
    result = df.copy()
    result["feedback_similarity"] = feedback_scores
    result["is_starred"] = False
    result["re_ranked_score"] = re_ranked_score
    result.iloc[starred_positions, result.columns.get_loc("is_starred")] = True

    # Sort the result by starred status and re-ranked score
    return result.sort_values(["is_starred", "re_ranked_score"],ascending=[False, False])

    

In [18]:
starred_positions = [1]  
candidate_embeddings = model.encode(results["concise_job_title"].fillna("Other").tolist(), normalize_embeddings=True, show_progress_bar=True)

result = candidate_feedback_similarity(results, candidate_embeddings, starred_positions)
result

Batches: 100%|██████████| 1/1 [00:00<00:00, 115.76it/s]


,id,job_title,concise_job_title,job_sector,city,region,seniority,fit,concise_similarity,original_similarity,sector_match,feedback_similarity,is_starred,re_ranked_score
79,80,junior mes engineer information systems,Data Engineer,technology,Myrtle Beach,South Carolina Area,junior,0.719935,0.740648,0.507405,1.0,1.0,True,0.803954
85,86,information systems specialist and programmer ...,Data Engineer,technology,Gaithersburg,Maryland,junior,0.731827,0.740648,0.566867,1.0,0.8,False,0.752279
